# 2 · Building and running a graph

A graph record is a graph as data: its nodes, what each of their inputs gets, and what the graph takes and returns. An editor saves it as JSON. `Graph.compile` checks a record against a registry and gives you a `Graph`, and `graph.run()` runs it.

Each input of a node gets one of three things:

- `sources`, the edges into it;
- a `constant`, the same on every run;
- a `graph_input`, a value the caller gives when the run starts, under a name the record declares in its `inputs`.

An input that gets none of them keeps its function's default.

In [ ]:
from conductor import Graph, GraphRecord, NodeRegistry, Series, make_graph

registry = NodeRegistry(category="text")


@registry.node("echo", title="Echo", description="Hands back the text it is given.")
def echo(text: str) -> str:
    return text


@registry.node("upper", title="Uppercase", description="Turns a text into capital letters.")
def upper(text: str) -> str:
    return text.upper()


@registry.node("pair", title="Pair", description="Joins two texts.")
def pair(a: str, b: str, separator: str = " + ") -> str:
    return f"{a}{separator}{b}"


@registry.node("words", title="Words", description="Gives the words of a text.")
def words(text: str) -> Series[str]:
    return text.split()


@registry.node("join-all", title="Join all", description="Joins every text it is given.")
def join_all(texts: Series[str]) -> str:
    return " ".join(texts)

## Build the graph

```text
  hello ──> shout ──┐
                    ├──> pair
  world ────────────┘
```

Each node has an id of its own in the graph, and `node` names the node it uses: `hello` and `world` both use `echo`. `GraphRecord` is the record's model; `Graph.compile` also takes the plain JSON, as text or as a dict.

In [ ]:
record = GraphRecord.model_validate({
    "inputs": {"hello": {"default": "hello"}, "world": {"default": "world"}},
    "nodes": {
        "hello": {"node": "echo", "inputs": {"text": {"graph_input": "hello"}}},
        "world": {"node": "echo", "inputs": {"text": {"graph_input": "world"}}},
        "shout": {"node": "upper", "inputs": {"text": {"sources": [{"node": "hello", "output": "result"}]}}},
        "pair": {"node": "pair", "inputs": {
            "a": {"sources": [{"node": "shout", "output": "result"}]},
            "b": {"sources": [{"node": "world", "output": "result"}]},
            "separator": {"constant": " & "},
        }},
    },
    "outputs": {"result": {"node": "pair", "output": "result"}},
})

graph = Graph.compile(record, registry)
print("runnable:", graph.is_runnable)

## Ask the compiled graph

`graph.describe()` is the whole compiled graph as one model, for an editor to draw: each node's inputs and outputs, with their types, and its `per_item_of`, the series it runs once per item of (`None` for a node that runs once). `model_dump(mode="json")` writes it as JSON.

In [ ]:
described = graph.describe()
print("pair takes:", [(each.port.name, each.port.type) for each in described.nodes["pair"].inputs])
print("runs:", {node_id: node.per_item_of or "once" for node_id, node in described.nodes.items()})

## Problems, not exceptions

A broken graph still compiles. Each problem has a stable `type`, and a `loc`: the node, then the input it is on. Running a graph with a problem raises `InvalidCallError`.

In [ ]:
broken = Graph.compile({"nodes": {"shout": {"node": "upper", "inputs": {"text": {"sources": [{"node": "ghost", "output": "result"}]}}}}}, registry)
print("runnable:", broken.is_runnable)
for problem in broken.problems:
    print(f"  {problem.type} on {problem.loc}: {problem.msg}")

## Run it

`run.results` is `{node_id: {output: value}}`.

In [ ]:
run = graph.run()
for node_id, outputs in run.results.items():
    print(f"  {node_id}: {outputs['result']!r}")

## Fill its inputs, read its outputs

A graph declares what it takes and returns, as a function declares its parameters and its return value. `hello` and `world` are its graph inputs, each with a default, and `result` is its graph output. A caller gives graph inputs as one mapping, by name, and `run.outputs` is what the graph returns.

In [ ]:
print("takes:  ", list(graph.inputs))
print("returns:", list(graph.outputs))

graph.run({"hello": "goodbye"}).outputs

## Watch the events

`graph.with_inputs()` makes a run without starting it, and `run.stream()` runs it, yielding an event as each node starts and ends. It calls the nodes the same way `run()` does.

In [ ]:
async for event in graph.with_inputs().stream():
    match event.type:
        case "node_start":
            print(f"  start     {event.loc[-1]}")
        case "node_complete":
            print(f"  complete  {event.loc[-1]}: ran {event.count.finished} time(s)")
        case "run_end":
            print(f"  done      {event.run.status}")
        case _:
            print(f"  {event.type}")

## Once per item

`words` returns a `Series[str]`: many texts. `upper` takes one text, so wired to `words` it runs once per word. `join-all` takes a `Series[str]`, so it runs once and gets every result, in order. `graph.describe()` says so before any run.

`make_graph` writes a record from Python. With no `edges=` the nodes form a chain, each under its own id. It declares the graph's inputs and outputs too: each required input that gets nothing, here `text`, and each output no edge leaves, here `result`.

In [ ]:
items = Graph.compile(make_graph(words, upper, join_all), registry)
for node_id, node in items.describe().nodes.items():
    print(f"  {node_id}: {node.per_item_of or 'once'}")

async for event in items.with_inputs({"text": "one two three"}).stream():
    if event.type == "node_progress":
        print(f"  {event.loc[-1]}: {event.count.finished} of {event.count.total}")
    elif event.type == "run_end":
        print("  upper:   ", event.run.results["upper"]["result"])
        print("  join-all:", event.run.results["join-all"]["result"])

## Save it

A `GraphRecord` reads back the JSON it writes. Fields left empty are left out.

In [ ]:
text = record.model_dump_json()
print(text)
assert GraphRecord.model_validate_json(text) == record